# ইমেজ জেনারেশন অ্যাপ্লিকেশন তৈরি করা (OpenAI)

LLM গুলোর মধ্যে কেবল টেক্সট জেনারেশনই নয় আরও অনেক কিছু রয়েছে। আপনি টেক্সট বর্ণনা থেকে ইমেজও তৈরি করতে পারেন। ইমেজ একটি মোডালিটি হিসাবে মেডটেক, স্থাপত্য, পর্যটন, গেম ডেভেলপমেন্ট, মার্কেটিং এবং আরও অনেক ক্ষেত্রে কার্যকর। এই পাঠে আমরা আজকের **GPT Image** মডেলসমূহ নিয়ে কাজ করব OpenAI প্ল্যাটফর্মে।

## শেখার লক্ষ্য

এই পাঠের শেষে আপনি সক্ষম হবেন:

- ব্যাখ্যা করতে যে ইমেজ জেনারেশন কী এবং কোথায় এটি উপযোগী।
- `gpt-image` মডেল পরিবারের সাথে পরিচিত হওয়া এবং এটি কীভাবে পুরোনো DALL·E মডেল থেকে পৃথক তা বোঝা।
- একটি ইমেজ জেনারেশন অ্যাপ্লিকেশন তৈরি করা এবং ইমেজ সম্পাদনা করা।

## ইমেজ জেনারেশন কী?

ইমেজ জেনারেশন মডেলগুলি একটি টেক্সট প্রম্পট থেকে ছবি তৈরি করে। আধুনিক মডেল যেমন `gpt-image` প্রশিক্ষণের সময় টেক্সট ও ইমেজের মধ্যে সম্পর্ক শেখে, তারপর ধাপে ধাপে র্যান্ডম শব্দোচ্চারণকে আপনার প্রম্পটের সাথে মিল সঙ্গত একটি ছবিতে পরিণত করে।

দুইটি পরিচিত ইমেজ মডেল পরিবারের নাম হলো:

- **`gpt-image` (OpenAI)** — এই পাঠে ব্যবহৃত বর্তমান প্রজন্ম। এটি টেক্সট-টু-ইমেজ জেনারেশন এবং ইমেজ এডিটিং (মাস্ক দিয়ে ইনপেইন্টিং) সমর্থন করে।
- **Midjourney** — একটি জনপ্রিয় তৃতীয় পক্ষের মডেল যার নিজস্ব সার্ভিস এবং Discord-ভিত্তিক ওয়ার্কফ্লো রয়েছে।

> পুরনো OpenAI ইমেজ মডেলসমূহ — **DALL·E 2** এবং **DALL·E 3** — লেগ্যাসি। DALL·E 3 আর নতুন ডিপ্লয়মেন্টের জন্য উপলব্ধ নেই, এবং `create_variation` এর মতো ফিচার গুলো কেবল DALL·E 2-তে ছিল। নতুন অ্যাপ্লিকেশনের জন্য `gpt-image` মডেল ব্যবহার করুন।

> **গুরুত্বপূর্ণ:** `gpt-image` মডেলগুলি তৈরি করা ছবি **base64** (`b64_json`) হিসেবে রিটার্ন করে, URL হিসেবে নয়। আপনার কোড base64 স্ট্রিংকে বাইটসে ডিকোড করে পরে সংরক্ষণ করে — ডাউনলোড করার জন্য কোনো ইমেজ URL থাকে না।


## আপনার প্রথম ইমেজ জেনারেশন অ্যাপ্লিকেশন তৈরি করা

তাহলে একটি ইমেজ জেনারেশন অ্যাপ্লিকেশন তৈরি করতে কী কী লাগবে? আপনার নিচের লাইব্রেরিগুলো দরকার হবে:

- **python-dotenv**, এই লাইব্রেরি ব্যবহার করার ব্যাপারে জোরালো ভাবে সুপারিশ করা হয়, যেন আপনি আপনার সিক্রেটগুলো কোড থেকে দূরে *.env* ফাইলে রাখতে পারেন।
- **openai**, এই লাইব্রেরি ব্যবহার করবেন OpenAI API-এর সাথে ইন্টারঅ্যাক্ট করার জন্য।
- **pillow**, পাইথনে ইমেজ নিয়ে কাজ করার জন্য।


1. একটি *.env* নামক ফাইল তৈরি করুন এবং তাতে নিচের বিষয়বস্তু লিখুন:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. উপরের লাইব্রেরিগুলো *requirements.txt* নামে একটি ফাইলে নিচের মত সংগ্রহ করুন:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. এরপর, ভার্চুয়াল এনভায়রনমেন্ট তৈরি করুন এবং লাইব্রেরিগুলো ইনস্টল করুন:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> উইন্ডোজের জন্য, আপনার ভার্চুয়াল এনভায়রনমেন্ট তৈরি এবং সক্রিয় করতে নিম্নলিখিত কমান্ডগুলি ব্যবহার করুন:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. *app.py* নামে একটি ফাইলে নিম্নলিখিত কোড যোগ করুন:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # dotenv আমদানি করুন
    dotenv.load_dotenv()

    # OpenAI অবজেক্ট তৈরি করুন (আপনার .env থেকে OPENAI_API_KEY পড়ে)
    client = openai.OpenAI()


    try:
        # ইমেজ জেনারেশন API ব্যবহার করে একটি ছবি তৈরি করুন
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # এখানে আপনার প্রম্পট টেক্সট প্রবেশ করুন
            size='1024x1024',
            n=1
        )
        # সংরক্ষিত ছবির জন্য ডিরেক্টরি নির্ধারণ করুন
        image_dir = os.path.join(os.curdir, 'images')

        # যদি ডিরেক্টরি না থাকে, তবে তা তৈরি করুন
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # ইমেজ পাথ ইনিশিয়ালাইজ করুন (দ্রষ্টব্য ফাইল টাইপ png হতে হবে)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image মডেলগুলি ছবিকে base64 (b64_json) হিসেবে ফেরত দেয়, URL হিসেবে নয়
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # ডিফল্ট ইমেজ ভিউয়ারে ছবি প্রদর্শন করুন
        image = Image.open(image_path)
        image.show()

    # এক্সসেপশন ধরুন
    except openai.BadRequestError as err:
        print(err)

    ```

চলুন এই কোড ব্যাখ্যা করি:

- প্রথমে, আমরা প্রয়োজনীয় লাইব্রেরিগুলি ইমপোর্ট করি, যার মধ্যে রয়েছে OpenAI লাইব্রেরি, dotenv লাইব্রেরি, base64 মডিউল, এবং Pillow লাইব্রেরি।

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- এর পরে, আমরা ক্লায়েন্ট তৈরি করি, যা আপনার ``.env`` থেকে API কী পড়ে।

    ```python
    # OpenAI অবজেক্ট তৈরি করুন
    client = openai.OpenAI()
    ```

- এর পর, আমরা ইমেজ তৈরি করি:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # আপনার প্রম্পট টেক্সট এখানে লিখুন
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` মডেলগুলো একটি **base64** স্ট্রিং `data[0].b64_json` হিসেবে ছবি ফিরিয়ে দেয়। আমরা এটিকে বাইটে ডিকোড করি এবং একটি ফাইলে লিখি — ডাউনলোড করার জন্য কোনও URL নেই।

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- সর্বশেষে, আমরা ছবিটি খুলে স্ট্যান্ডার্ড ইমেজ ভিউয়ার ব্যবহার করে দেখাই:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### ছবির জেনারেশনের আরও বিবরণ

চলুন `images.generate` এর প্যারামিটারগুলি দেখি:

- **model**, ছবির জন্য ব্যবহৃত মডেল (যেমন `gpt-image-1`)।
- **prompt**, ছবিটি তৈরি করতে ব্যবহৃত টেক্সট প্রম্পট। এখানে এটি "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils"।
- **size**, তৈরিকৃত ছবির আকার (যেমন `1024x1024`, `1536x1024`, `1024x1536`, অথবা `"auto"`)।
- **n**, তৈরিকৃত ছবির সংখ্যা। এখানে আমরা একটি তৈরি করি।

> ইমেজ মডেলগুলো `temperature` প্যারামিটার ব্যবহার করে না — এটি টেক্সট জেনারেশনের নিয়ন্ত্রণ। বৈচিত্র্য পেতে, আবার API কল করুন; বৈচিত্র্য কমাতে আপনার প্রম্পটকে আরও নির্দিষ্ট করুন।

## চিত্র তৈরির অতিরিক্ত ক্ষমতা

আপনি দেখেছেন কিভাবে কয়েক লাইনের পাইথন দিয়ে একটি ছবি তৈরি করা যায়। `gpt-image` মডেলগুলো একটি অবস্থিত ছবি **সম্পাদনা** করতেও পারে। একটি ছবি, ঐচ্ছিক **মাস্ক** (যা পরিবর্তন করার এলাকা চিহ্নিত করে), এবং একটি প্রম্পট প্রদান করে আপনি ছবির অংশ পরিবর্তন করতে পারেন — উদাহরণস্বরূপ, আমাদের খরগোশটিতে একটি টুপি যোগ করুন।

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# এডিটগুলি বেস64 হিসাবেও ফিরিয়ে দেওয়া হয়
edited_image = base64.b64decode(response.data[0].b64_json)
```

বেস ছবিতে কেবল খরগোশ রয়েছে; চূড়ান্ত ছবিতে খরগোশের উপরে টুপি রয়েছে।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
